In [1]:
# gnn_dgi_clustering_3d_multi_confeggn_rep.py
import os
import random
import math
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import global_mean_pool
# SchNet
from torch_geometric.nn.models import SchNet
# DGI
from torch_geometric.nn.models import DeepGraphInfomax
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.manifold import TSNE


In [2]:
# ---------- CONFIG ----------
INPUT_FILE = "combined_results_3groups_HT.xlsx"
SMILES_COL = "SMILES"
OUT_DIR = "results_schnet_dgi_multi3d_egnn"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DGI / training
HIDDEN_DIM = 128
BATCH_SIZE = 32
DGI_EPOCHS = 60
LR = 1e-3

# multi-conformer
NUM_CONFS = 3          # 每个分子的构象数（调小用于调试）
EMBEDDING_POOL = "mean"  # 如何聚合构象嵌入: "mean" or "median"

# clustering
USE_HDBSCAN_IF_AVAILABLE = True
DBSCAN_EPS_START = 0.8
DBSCAN_EPS_STEP = 0.2
DBSCAN_EPS_MAX = 2.0

# reassign noise?
REASSIGN_NOISE = True
# ---------- helpers ----------
def safe_embed_optimize(mol, max_tries=3):
    """Embed molecule and optimize. Returns True if succeeded."""
    for _ in range(max_tries):
        try:
            # ETKDG with random seed
            params = AllChem.ETKDG()
            params.randomSeed = random.randint(1, 2**20)
            cid = AllChem.EmbedMolecule(mol, params)
            if cid < 0:
                continue
            # optimize this conformer
            AllChem.UFFOptimizeMolecule(mol, confId=cid)
            return True
        except Exception:
            continue
    return False

# try to import EGNNConv (prefer torch_geometric's if available)
has_egnn = False
try:
    from torch_geometric.nn import EGNNConv
    print("Found EGNNConv in torch_geometric.nn")
    has_egnn = True
except Exception:
    try:
        # try egnn_pytorch
        from egnn_pytorch import EGNN
        print("Found EGNN in egnn_pytorch")
        has_egnn = True
    except Exception:
        print("EGNN not available - will use SchNet encoder.")


Found EGNN in egnn_pytorch


In [4]:

    # ---------- Step 1. Read SMILES ----------
df = pd.read_excel(INPUT_FILE)
smiles_list_all = df[SMILES_COL].dropna().tolist()
print("Total SMILES in file:", len(smiles_list_all))


Total SMILES in file: 1431


In [22]:
# ---------- Step 2. Build multi-conformer graph list ----------
# We'll build a list of conformer graphs with mapping to molecule index.
conformer_graphs = []   # list of Data objects (each a conformer)
conformer_mol_idx = []  # parallel list mapping conformer -> mol index
valid_smiles = []       # list of SMILES that produced at least one conformer

print("Generating conformers (this may take a while)...")
for mol_idx, smi in enumerate(tqdm(smiles_list_all, desc="SMILES")):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    mol_h = Chem.AddHs(mol)
    # try multiple conformers with EmbedMultipleConfs for efficiency
    try:
        # use ETKDG to generate NUM_CONFS conformers
        params = AllChem.ETKDG()
        params.randomSeed = SEED + mol_idx
        cids = AllChem.EmbedMultipleConfs(mol_h, numConfs=NUM_CONFS, params=params)
        if len(cids) == 0:
            # fallback: try single embed
            ok = safe_embed_optimize(mol_h)
            if not ok:
                continue
            cids = [0]
        # optimize all conformers (ULL Optimize)
        AllChem.UFFOptimizeMoleculeConfs(mol_h)
    except Exception:
        # retry single embed
        ok = safe_embed_optimize(mol_h)
        if not ok:
            continue
        cids = [0]

    # extract conformers
    added_any = False
    for cid in cids:
        try:
            conf = mol_h.GetConformer(cid)
            pos = []
            z = []
            for atom in mol_h.GetAtoms():
                p = conf.GetAtomPosition(atom.GetIdx())
                pos.append([p.x, p.y, p.z])
                z.append(atom.GetAtomicNum())
            num_atoms = len(z)
            data = Data(z=torch.tensor(z, dtype=torch.long),
                        pos=torch.tensor(pos, dtype=torch.float),
                        batch=torch.zeros(num_atoms, dtype=torch.long))
            conformer_graphs.append(data)
            conformer_mol_idx.append(len(valid_smiles))  # temp index; will append smiles below once at least one conf
            added_any = True
        except Exception:
            continue
    if added_any:
        valid_smiles.append(smi)  # map this mol_idx -> new index in valid_smiles

# Note: conformer_mol_idx currently uses index into valid_smiles being built sequentially.
n_mols = len(valid_smiles)
n_conformers = len(conformer_graphs)
print(f"Built {n_conformers} conformers for {n_mols} valid molecules.")

if n_mols == 0:
    raise SystemExit("No valid 3D molecules. Check SMILES and RDKit.")


Generating conformers (this may take a while)...


SMILES: 100%|██████████████████████████████████████████████████████████████████████| 1431/1431 [06:49<00:00,  3.49it/s]

Built 4293 conformers for 1431 valid molecules.


In [23]:
# ---------- Step 3. Encoder selection ----------
# SchNetEncoder (default)
class SchNetEncoder(torch.nn.Module):
    def __init__(self, hidden_channels=128, num_filters=128, num_interactions=6):
        super().__init__()
        self.schnet = SchNet(
            hidden_channels=hidden_channels,
            num_filters=num_filters,
            num_interactions=num_interactions,
            num_gaussians=50,
            cutoff=10.0,      # 截断距离，Å
            max_num_neighbors=32
        )

    def forward(self, z, pos, batch):
        x = self.schnet(z, pos, batch)
        return x

# Simple EGNN wrapper if EGNNConv exists
EGNN_USABLE = False
if has_egnn:
    try:
        # try torch_geometric's EGNNConv (if available)
        from torch_geometric.nn import EGNNConv
        EGNN_USABLE = True
        class EGNNEncoder(torch.nn.Module):
            def __init__(self, node_emb_dim=64, n_layers=3):
                super().__init__()
                # node embedding layer from atomic number
                self.node_emb = torch.nn.Embedding(100, node_emb_dim)  # atomic numbers up to ~100
                self.convs = torch.nn.ModuleList()
                for _ in range(n_layers):
                    self.convs.append(EGNNConv(in_channels=node_emb_dim, edge_dim=None))
                self.lin = torch.nn.Linear(node_emb_dim, node_emb_dim)
            def forward(self, z, pos, batch):
                h = self.node_emb(z)  # per-atom embedding
                # EGNNConv probably uses (x, edge_index, edge_attr) signature in torch_geometric;
                # however EGNNConv from different versions may vary; we will use message passing only with positions
                # For safety, attempt iterative updates with zero edge_index (fully connected handled internally)
                for conv in self.convs:
                    try:
                        h, pos = conv(h, pos)  # some implementations accept (h, pos)
                    except TypeError:
                        # fallback: conv expects (x, edge_index, edge_attr)
                        h = conv(h, None, None)
                h = self.lin(h)
                return h
    except Exception:
        EGNN_USABLE = False

# choose encoder: prefer EGNN if usable, else SchNet
if EGNN_USABLE:
    encoder = EGNNEncoder(node_emb_dim=HIDDEN_DIM//2, n_layers=3).to(DEVICE)
    print("Using EGNNEncoder (if implemented in environment).")
else:
    encoder = SchNetEncoder(hidden_channels=HIDDEN_DIM).to(DEVICE)
    print("Using SchNetEncoder.")

# ---------- DGI summary & corruption ----------

def summary_func(z, batch=None):
    """
    z: node embeddings [N_nodes, hidden_dim]
    batch: batch indices
    """
    if batch is None:
        # 返回节点均值作为全局 summary
        s = torch.sigmoid(z.mean(dim=0))   # [hidden_dim]
    else:
        # 每个图先池化，然后再求 batch 均值
        s = torch.sigmoid(global_mean_pool(z, batch).mean(dim=0))  # [hidden_dim]
    return s

# wrapper 保证 kwargs 兼容
def summary_wrapper(*args, **kwargs):
    z = args[0]
    batch = kwargs.get("batch", None)
    return summary_func(z, batch)

def corruption(z, pos, batch):
    # shuffle node features across batch
    idx = torch.randperm(z.size(0))
    return z[idx], pos, batch

dgi = DeepGraphInfomax(
    hidden_channels=HIDDEN_DIM,
    encoder=encoder,
    summary=summary_wrapper,
    corruption=corruption
).to(DEVICE)

optimizer = torch.optim.Adam(dgi.parameters(), lr=LR)


Using SchNetEncoder.


In [24]:
# ---------- Step 4. DGI training over conformers ----------
print("Start DGI training on conformers...")
dgi.train()
conformer_loader = DataLoader(conformer_graphs, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(1, DGI_EPOCHS + 1):
    total_loss = 0.0
    valid_batches = 0

    for batch in conformer_loader:
        batch = batch.to(DEVICE)

        # 检查 x 和 pos 是否存在
        if batch.x is None or batch.pos is None:
            continue  # 跳过无效 batch

        node_features = batch.x
        pos = batch.pos

        # 确保 node_features 是二维 [num_nodes, node_dim]
        if node_features.dim() == 1:
            node_features = node_features.unsqueeze(-1)

        # Forward pass
        try:
            pos_z, neg_z, summary = dgi(node_features, pos, batch.batch)
        except Exception as e:
            print("DGI forward error, skipping batch:", e)
            continue

        # summary 必须是一维向量
        if summary.dim() > 1:
            summary = summary.squeeze(0)

        # 计算 DGI loss
        loss = dgi.loss(pos_z, neg_z, summary)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item()) * batch.num_graphs
        valid_batches += batch.num_graphs

    if valid_batches == 0:
        print("No valid batches this epoch, skipping...")
        continue

    avg_loss = total_loss / valid_batches
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} avg_loss={avg_loss:.6f}")

print("✅ DGI training finished.")


Start DGI training on conformers...
No valid batches this epoch, skipping...


C:\Users\yhh19\AppData\Local\Temp\ipykernel_18280\1344809797.py:4: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  conformer_loader = DataLoader(conformer_graphs, batch_size=BATCH_SIZE, shuffle=True)


No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches this epoch, skipping...
No valid batches

In [25]:
# ---------- Step 5. Extract conformer embeddings and average per molecule ----------
encoder.eval()
conformer_embeddings = []
batch_mol_idx_list = []

with torch.no_grad():
    eval_loader = DataLoader(conformer_graphs, batch_size=BATCH_SIZE, shuffle=False)
    for batch_idx, batch in enumerate(eval_loader):
        batch = batch.to(DEVICE)

        # 确保节点特征存在
        if not hasattr(batch, 'z') or batch.z is None:
            continue

        # Node embeddings
        node_z = encoder(batch.z, batch.pos, batch.batch)  # [num_nodes_in_batch, hidden_dim]

        # Graph-level embeddings (每个 conformer)
        graph_z = global_mean_pool(node_z, batch.batch)    # [num_conformers_in_batch, hidden_dim]

        conformer_embeddings.append(graph_z.cpu().numpy())

        # 保存 batch 对应的 molecule idx
        # 注意这里要对应 DataLoader 返回的每个 conformer 的 index
        batch_mol_idx_list.extend([conformer_mol_idx[i] for i in range(batch_idx*BATCH_SIZE, batch_idx*BATCH_SIZE + batch.num_graphs)])

# 合并所有 conformer embeddings
conformer_embeddings = np.vstack(conformer_embeddings)  # [n_conformers_total, hidden_dim]
print("Conformer embeddings shape:", conformer_embeddings.shape)

# Molecule-level 聚合
mol_embs = np.zeros((n_mols, conformer_embeddings.shape[1]), dtype=float)
counts = np.zeros(n_mols, dtype=int)

for i, mol_idx in enumerate(batch_mol_idx_list):
    mol_embs[mol_idx] += conformer_embeddings[i]
    counts[mol_idx] += 1

# 避免除以 0
for i in range(n_mols):
    if counts[i] > 0:
        if EMBEDDING_POOL == "mean":
            mol_embs[i] /= counts[i]
        elif EMBEDDING_POOL == "median":
            inds = [j for j, m in enumerate(batch_mol_idx_list) if m == i]
            mol_embs[i] = np.median(conformer_embeddings[inds, :], axis=0)
    else:
        mol_embs[i] = np.zeros(conformer_embeddings.shape[1])

print("Molecule-level embeddings shape:", mol_embs.shape)

# L2 normalize
embeddings_norm = mol_embs / np.linalg.norm(mol_embs, axis=1, keepdims=True)


C:\Users\yhh19\AppData\Local\Temp\ipykernel_18280\1219025842.py:7: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  eval_loader = DataLoader(conformer_graphs, batch_size=BATCH_SIZE, shuffle=False)


RuntimeError: Expected index [1741] to be no larger than self [32] apart from dimension 0 and to be no larger size than src [32]

In [ ]:

# ---------- Step 6. Clustering (HDBSCAN preferred) ----------
try:
    import hdbscan
    has_hdbscan = True
except Exception:
    has_hdbscan = False

labels_final = None
n_clusters_final = 0
if has_hdbscan and USE_HDBSCAN_IF_AVAILABLE:
    print("Using HDBSCAN for clustering...")
    min_cluster_size = max(5, int(0.01 * len(embeddings_norm)))
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, metric="euclidean")
    labels = clusterer.fit_predict(embeddings_norm)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    labels_final = labels
    n_clusters_final = n_clusters
    print("HDBSCAN clusters:", n_clusters)
else:
    print("HDBSCAN not available or disabled — using KMeans/DBSCAN fallback.")
    # try DBSCAN auto-eps first
    eps = DBSCAN_EPS_START
    labels_db = None
    while eps <= DBSCAN_EPS_MAX:
        labels_try = DBSCAN(eps=eps, min_samples=2).fit_predict(embeddings_norm)
        n_clusters_try = len(set(labels_try[labels_try != -1]))
        print(f"DBSCAN eps={eps:.2f} -> valid clusters={n_clusters_try}")
        if n_clusters_try >= 2:
            labels_db = labels_try
            break
        eps += DBSCAN_EPS_STEP
    if labels_db is not None:
        labels_final = labels_db
        n_clusters_final = len(set(labels_db[labels_db != -1]))
    else:
        # grid KMeans by silhouette
        best_score = -1
        best_labels = None
        best_k = None
        for k in range(2, min(12, len(embeddings_norm)-1)):
            km = KMeans(n_clusters=k, random_state=SEED)
            lab = km.fit_predict(embeddings_norm)
            try:
                s = silhouette_score(embeddings_norm, lab)
            except Exception:
                s = -1
            print(f"KMeans k={k} silhouette={s:.4f}")
            if s > best_score:
                best_score = s
                best_labels = lab
                best_k = k
        labels_final = best_labels
        n_clusters_final = best_k
        print(f"Selected KMeans k={best_k}, silhouette={best_score:.4f}")

print("Final cluster count (excluding noise):", n_clusters_final)


In [ ]:
# ---------- Step 7. Optionally reassign noise with KMeans ----------
def assign_noise_with_kmeans(labels, embeddings, n_clusters_target):
    labels = labels.copy()
    noise_mask = labels == -1
    if noise_mask.sum() == 0:
        return labels
    base_mask = ~noise_mask
    if base_mask.sum() == 0:
        km = KMeans(n_clusters=n_clusters_target, random_state=SEED)
        labels_full = km.fit_predict(embeddings)
        return labels_full
    km = KMeans(n_clusters=n_clusters_target, random_state=SEED)
    km.fit(embeddings[base_mask])
    assigned = km.predict(embeddings[noise_mask])
    labels[noise_mask] = assigned
    return labels

if REASSIGN_NOISE and labels_final is not None and (-1 in set(labels_final)):
    print("Reassigning noise points using KMeans...")
    # determine target cluster count
    target_k = n_clusters_final if (n_clusters_final and n_clusters_final>0) else max(2, int(0.01*len(embeddings_norm)))
    labels_final = assign_noise_with_kmeans(labels_final, embeddings_norm, target_k)


In [ ]:
# ---------- Step 8. Cluster evaluation ----------
unique_labels = set(labels_final)
n_effective = len(unique_labels) - (1 if -1 in unique_labels else 0)
if n_effective < 2:
    print("Warning: less than 2 effective clusters; skipping metrics.")
else:
    try:
        sil = silhouette_score(embeddings_norm, labels_final)
        ch = calinski_harabasz_score(embeddings_norm, labels_final)
        db = davies_bouldin_score(embeddings_norm, labels_final)
        print(f"Silhouette: {sil:.4f}, Calinski-Harabasz: {ch:.4f}, Davies-Bouldin: {db:.4f}")
    except Exception as e:
        print("Metrics computation failed:", e)


In [28]:
# ---------- Step 9. Choose representative molecule per cluster ----------
labels_arr = np.array(labels_final)
representative_idx = {}
centroids = {}
for lab in sorted(set(labels_arr)):
    if lab == -1:
        continue
    inds = np.where(labels_arr == lab)[0]
    cluster_embs = embeddings_norm[inds]
    centroid = cluster_embs.mean(axis=0)
    centroids[lab] = centroid
    # choose closest molecule (min Euclidean distance)
    dists = np.linalg.norm(cluster_embs - centroid[None, :], axis=1)
    rep_local_idx = np.argmin(dists)
    rep_global_idx = inds[rep_local_idx]
    representative_idx[lab] = rep_global_idx

✅ 分子嵌入维度: (1431, 64)
✅ 归一化嵌入维度: (1431, 64)


In [29]:
# ---------- Step 10. Save results ----------
# Build DataFrame for molecules (valid_smiles order)
df_out = pd.DataFrame({
    "SMILES": valid_smiles,
    "Cluster": labels_final
})
# mark representative molecules
df_out["IsRepresentative"] = False
for lab, midx in representative_idx.items():
    df_out.loc[midx, "IsRepresentative"] = True

xlsx_out = os.path.join(OUT_DIR, "schnet_dgi_multi3d_clusters_reps.xlsx")
df_out.to_excel(xlsx_out, index=False)
print("Saved clusters & representatives to:", xlsx_out)

# Save embeddings and centroids
np.save(os.path.join(OUT_DIR, "molecule_embeddings.npy"), mol_embs)
np.save(os.path.join(OUT_DIR, "molecule_embeddings_norm.npy"), embeddings_norm)
# centroid dict -> array
centroid_arr = np.vstack([centroids[k] for k in sorted(centroids.keys())]) if centroids else np.empty((0, mol_embs.shape[1]))
np.save(os.path.join(OUT_DIR, "cluster_centroids.npy"), centroid_arr)


DBSCAN eps=0.80, 有效簇数=1
DBSCAN eps=1.00, 有效簇数=1
DBSCAN eps=1.20, 有效簇数=1
DBSCAN eps=1.40, 有效簇数=1
DBSCAN eps=1.60, 有效簇数=1
DBSCAN eps=1.80, 有效簇数=1
DBSCAN eps=2.00, 有效簇数=1
⚠️ DBSCAN 未找到足够簇，使用 KMeans 自动簇数选择
✅ 最终簇数: 2


In [30]:
# ---------- Step 11. t-SNE visualization ----------
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30)
emb_2d = tsne.fit_transform(embeddings_norm)

plt.figure(figsize=(8,6))
plt.scatter(emb_2d[:,0], emb_2d[:,1], c=labels_final, cmap="tab20", s=40, edgecolor='k')
plt.title(f"3D DGI clustering (n_clusters={n_clusters_final})")
plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
plt.tight_layout()
png_out = os.path.join(OUT_DIR, "schnet_dgi_multi3d_tsne.png")
plt.savefig(png_out, dpi=300)
plt.show()
print("Saved t-SNE to:", png_out)

# ---------- Step 12. Print representative SMILES per cluster ----------
print("\nRepresentative molecules per cluster:")
for lab, midx in representative_idx.items():
    print(f"Cluster {lab}: SMILES = {valid_smiles[midx]} (index {midx})")

Silhouette Score: 0.3519
Calinski-Harabasz Score: 1093.4626
Davies-Bouldin Score: 1.0555
